In [ ]:
# this is the file created to embed the titles and get the faiss index files
# the paths are specific to google colabs environment because it has access to T4 GPU and I havent properly set up my Local GPU in Linux
import pandas as pd
import numpy as np
import re
import os
from html import unescape

# Read the unified raw clean data (pre-cleaned by clean_get_titles_csv.ipynb)
# df_collection = pd.read_csv("../data/raw_data_clean.csv")
df_collection = pd.read_csv("/content/raw_data_clean.csv", engine='python')

# Reset index so that the row index matches the FAISS vector index (faiss_id)
df_collection = df_collection.reset_index(drop=True)
print(f"Loaded {len(df_collection)} rows.")

Loaded 113681 rows.


In [3]:
# Helper to convert comma-separated strings to list of tags/genres
def split_to_list(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in value.split(",") if item.strip()]

# Convert tags and genres to list type for unique lists & formatting
df_collection["tags_list"] = df_collection["tags"].apply(split_to_list)
df_collection["genres_list"] = df_collection["genres"].apply(split_to_list)

In [5]:
# Extract unique tags across the entire collection
unique_tags = set()
for tag_list in df_collection['tags_list']:
    for tag in tag_list:
        unique_tags.add(tag.strip())

unique_tags = sorted(list(unique_tags))
print(f"Total unique tags: {len(unique_tags)}")
# with open("../data/unique_tags.txt", "w") as f:
with open("/content/unique_tags.txt", "w") as f:
    for tag in unique_tags:
        f.write(f"{tag}\n")

# Extract unique genres across the entire collection
unique_genres = set()
for genre_list in df_collection['genres_list']:
    for genre in genre_list:
        unique_genres.add(genre.strip())

unique_genres = sorted(list(unique_genres))
print(f"Total unique genres: {len(unique_genres)}")
# with open("../data/unique_genres.txt", "w") as f:
with open("/content/unique_genres.txt", "w") as f:
    for genre in unique_genres:
        f.write(f"{genre}\n")

Total unique tags: 424
Total unique genres: 19


In [7]:
# Create combined text column representing the item for embeddings
df_collection["combined_text"] = df_collection.apply(
    lambda row: (
        f"Title: {row['title']}. "
        f"Genres: {', '.join(row['genres_list'])}. "
        f"Tags: {', '.join(row['tags_list'])}. "
        f"Description: {row['description']}."
    ),
    axis=1
)
print("Sample combined text:")
print(df_collection["combined_text"].iloc[0])

Sample combined text:
Title: Magical Kanan. Genres: Drama, Fantasy, Mahou Shoujo, Supernatural. Tags: Magic, Henshin, School, Female Protagonist, Shapeshifting, Cute Girls Doing Cute Things, Kuudere. Description: Based on the erotic game by Terios.<br> <br> Five dangerous "seeds" have been stolen from their vault in the world of Evergreen and taken to Earth. Natsuki, an agent of the Queen, has been sent to retrieve them. However, he cannot do this alone and enlists the aid of Chihaya, a middle school student, who is "the only one" who can help him. With his aid she transforms into Magical Warrior Carmine in order to save the humans possessed by the seeds. Questions abound though, such as why Chihaya's mother appears to know all about this secret activity, why Chihaya remembers her father who reportedly died before her birth, and why she dreams of Evergreen, which she has never seen. <br><br> (Source: Anime News Network).


In [8]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 102.8 MB/s eta 0:00:00


In [9]:
import torch
from sentence_transformers import SentenceTransformer

# Detect T4 GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

print("Generating embeddings (this may take a few minutes)...")
embeddings = embedding_model.encode(
    df_collection['combined_text'].tolist(),
    show_progress_bar=True
)

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings (this may take a few minutes)...


Batches:   0%|          | 0/3553 [00:00<?, ?it/s]

Embeddings shape: (113681, 384)


In [11]:
import faiss

# Cast embeddings to float32
embeddings = embeddings.astype('float32', copy=False)

# Normalize embeddings for cosine similarity
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1e-12
embeddings = embeddings / norms

# Create FAISS index
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)
print("Vectors in index:", index.ntotal)

# Ensure output directory exists
os.makedirs("../data", exist_ok=True)

# Save the FAISS index
faiss.write_index(index, "/content/unified_index.faiss")

# Insert faiss_id as the first column for database mapping
if "faiss_id" not in df_collection.columns:
    df_collection.insert(0, "faiss_id", df_collection.index)
else:
    df_collection["faiss_id"] = df_collection.index

# Save the final metadata dataset
df_collection.to_csv("/content/cleaned_collections_with_combined_text.csv", index=False)

# Save raw embeddings array optionally
np.save("/content/embeddings.npy", embeddings)
print("Saved unified index, CSV, and raw embeddings successfully.")

Vectors in index: 113681
Saved unified index, CSV, and raw embeddings successfully.
